# 图像卷积

互相关运算

In [2]:
import torch
from torch import nn
from d2l import torch as d2l

def corr2d(X, K):
    """计算二维互相关运算"""
    h, w = K.shape  # 得到核的宽和高
    Y = torch.zeros((X.shape[0] - h + 1, X.shape[1] - w + 1))  # 输出
    for i in range(Y.shape[0]):
        for j in range(Y.shape[1]):
            Y[i, j] = (X[i: i + h, j:j + w] * K).sum()  # 计算核中每一单元与输入的每一个的乘积之和
    return Y

实现二维卷积层

In [3]:
class Conv2D(nn.Module):
    def __init__(self, kernel_size):
        super().__init__()
        self.weight = nn.Parameter(torch.rand(kernel_size))
        self.bias = nn.Parameter(torch.zeros(1))

    def forward(self, x):
        return corr2d(x, self.weight) + self.bias

# 学习卷积核

给定输入 `X` 和其经过某个隐藏核 `K` 互相关运算得到的输出 `Y`，将 `K` 作为 ground-truth。

构造一个 `Conv2D` 层，用梯度下降从 `<X,Y>` 中学习出 `K`。

首先构造训练数据

In [ ]:
# 构造输入 X：6×8 矩阵，左2列为1，中间4列为0，右2列为1
# 这样形成了两条竖边（列2处一条、列6处一条）
X = torch.ones((6, 8))
X[:, 2:6] = 0
X

In [ ]:
# 隐藏核 K：1×2 水平梯度核，用于检测竖边（相邻两列的差值）
K = torch.tensor([[1.0, -1.0]])

# 目标输出 Y = X 与 K 的互相关运算结果
Y = corr2d(X, K)
Y

In [ ]:
# 构造卷积层，输入通道=1，输出通道=1，核形状=(1, 2)
conv2d = nn.Conv2d(1, 1, kernel_size=(1, 2), bias=False)

# 将 X 和 Y 变形为 4D 张量：(批量大小, 通道数, 高, 宽)
X_2d = X.reshape((1, 1, 6, 8))
Y_2d = Y.reshape((1, 1, 6, 7))

lr = 3e-2  # 学习率

for i in range(15):
    Y_hat = conv2d(X_2d)                     # 前向传播
    loss = (Y_hat - Y_2d) ** 2               # 均方误差
    conv2d.zero_grad()                       # 清零梯度
    loss.sum().backward()                    # 反向传播
    # 手动 SGD：W = W - lr * grad
    conv2d.weight.data[:] -= lr * conv2d.weight.grad
    if (i + 1) % 3 == 0:
        print(f'epoch {i + 1:2d}, loss {loss.sum():.5f}')

In [ ]:
# 学到的核应该接近 [[1.0, -1.0]]
learned_K = conv2d.weight.data.reshape((1, 2))
print(f'隐藏核 K:       {K}')
print(f'学到的核 K\':    {learned_K}')

填充和步幅

在所有侧边都填充1个像素

In [4]:
import torch
from torch import nn

def  comp_conv2d(conv2d, X):
    X = X.reshape((1, 1) + X.shape)
    Y = conv2d(X)
    return Y.reshape(Y.shape[2:])

conv2d = nn.Conv2d(1, 1, kernel_size=3, padding=1)
X = torch.rand(size=(8, 8))
comp_conv2d(conv2d, X).shape

torch.Size([8, 8])

填充不同的高度和宽度

In [5]:
conv2d = nn.Conv2d(1, 1, kernel_size=(5, 3), padding=(2, 1))  # 上下填充两行，左右填充1列
comp_conv2d(conv2d, X).shape

torch.Size([8, 8])

将高度和宽度的步幅设置为2

In [6]:
conv2d = nn.Conv2d(1, 1, kernel_size=3, padding=1, stride=2)
comp_conv2d(conv2d, X).shape

torch.Size([4, 4])

In [7]:
conv2d = nn.Conv2d(1, 1, kernel_size=(3, 5), padding=(0, 1), stride=(3, 4))
comp_conv2d(conv2d, X).shape

torch.Size([2, 2])

# 1×1 卷积层

1×1 卷积指核尺寸为 $1 \times 1$ 的卷积。它不做空间模式识别，只在**通道维度**上做线性组合。

## 本质

对输入 $(C_{in}, H, W)$ 中的每个空间位置 $(h, w)$，取出 $C_{in}$ 维向量，用 $C_{out}$ 组 $C_{in}$ 维权值分别做内积，输出 $C_{out}$ 维。

```
位置 (h,w) 处:

  输入: [x₁, x₂, ..., x_{Cin}]                         ← 该位置所有通道的值
  核1:  [w₁¹,   w₂¹,   ..., w_{Cin}¹]    →   y₁
  核2:  [w₁²,   w₂²,   ..., w_{Cin}²]    →   y₂
  ...
  核C_out: [w₁^{Cout}, ...]               →   y_{Cout}

  输出: [y₁, y₂, ..., y_{Cout}]                         ← 输出该位置的所有通道
```

等价于：在每个空间位置上对通道做**全连接层**，权重在所有位置共享。

## 与全连接层的对比

| | 全连接层 | 1×1 卷积 |
|---|---|---|
| 输入形状 | $(N, C_{in})$ | $(C_{in}, H, W)$ |
| 操作 | 样本间独立，展平所有特征 | 空间位置独立，跨通道融合 |
| 空间信息 | 被打平丢失 | 保持 $H \times W$ |
| 参数量 | $C_{in} \times C_{out}$ | $C_{in} \times C_{out}$ |

## 用途

1. **升降维**：改变通道数，空间尺寸不变，参数极少
2. **增加非线性**：1×1 卷积 + 激活函数 = 每个位置的非线性变换
3. **跨通道融合**：与逐通道卷积（depthwise conv）搭配做通道混合
4. **计算瓶颈**：ResNet / Inception 中用 1×1 先降维再升维，减少大核卷积运算量

下面用代码演示

In [ ]:
# 多通道输入 → 多通道输出的 1×1 卷积
# 输入：3通道  ← 模拟 RGB 图像的3个通道
# 输出：2通道  ← 比如压缩后的表示

def corr2d_multi_in_out(X, K):
    """多输入通道 → 多输出通道的互相关（完整版卷积）"""
    return torch.stack([corr2d_multi_in(X, k) for k in K], 0)

def corr2d_multi_in(X, K):
    """单输出通道的多输入互相关：各通道分别做互相关再求和"""
    return sum(corr2d(x, k) for x, k in zip(X, K))

# 3 通道输入 (C=3, H=3, W=3)
X_multi = torch.arange(27, dtype=torch.float32).reshape(3, 3, 3)
print(f'输入形状: {X_multi.shape}')

# 1×1 卷积核 (C_out=2, C_in=3, 1, 1)
K_1x1 = torch.arange(6, dtype=torch.float32).reshape(2, 3, 1, 1)
print(f'核形状:   {K_1x1.shape}')

# 用互相关函数计算输出
Y_manual = corr2d_multi_in_out(X_multi, K_1x1)
print(f'输出形状: {Y_manual.shape}')

In [ ]:
# 验证：1×1 卷积 = 对每个空间位置做全连接（矩阵乘法）
# 取位置 (0, 0)，3 通道值 → 与核做线性变换 → 2 通道值

pixel_input  = X_multi[:, 0, 0]          # (3,)  -- 位置 (0,0) 的各通道值
weight_1x1   = K_1x1.reshape(2, 3)       # (2, 3) -- 两组权值
pixel_output = weight_1x1 @ pixel_input  # (2,)   -- 矩阵乘法

print(f'位置 (0,0) 输入 (3通道): {pixel_input}')
print(f'线性变换结果 (2通道):   {pixel_output}')
print(f'互相关输出 (0,0):       {Y_manual[:, 0, 0]}')
print(f'二者相等: {torch.allclose(pixel_output, Y_manual[:, 0, 0])}')

In [ ]:
# 用 PyTorch 内置 Conv2d 验证
conv_1x1 = nn.Conv2d(3, 2, kernel_size=1, bias=False)       # C_in=3, C_out=2
conv_1x1.weight.data = K_1x1                                 # 赋同样的权值
X_4d = X_multi.reshape(1, 3, 3, 3)                           # 加 batch 维度

Y_torch = conv_1x1(X_4d).squeeze(0)                          # (2, 3, 3)
print(f'PyTorch Conv2d 输出形状: {Y_torch.shape}')
print(f'手写互相关输出:         {Y_manual.shape}')
print(f'二者相等: {torch.allclose(Y_torch, Y_manual)}')

多输入多输出通道

In [8]:
# 多输入通道互相关运算
import torch
from d2l import torch as d2l

def corr2d_multi_in(X, K):
    return sum(d2l.corr2d(x, k) for x, k in zip(X, K))

In [10]:
# 多个通道的输出的互相关函数
def corr2d_multi_in_out(X, K):
    return torch.stack([corr2d_multi_in(X, k) for k in K], 0)

1x1卷积

In [ ]:
def corr2d_multi_in_out_1x1(X, K):
    c_i, h, w = X.shape
    c_o = K.shape[0]
    X = X.reshape((c_i, h * w))
    K = K.reshape((c_o, c_i))
    Y = torch.matmul(K, X)
    return Y.reshape((c_o, h, w))

X = torch.normal(0, 1, (3, 3, 3))
K = torch.normal(0, 1, (2, 3, 1, 1))